# simnibs-reader — usage examples

> No SimNIBS dependency needed — pure NIfTI-based reader.

In [1]:
import sys
sys.path.insert(0, '/Users/hippolyte.dreyfus/Documents/simnibs-reader')
import simnibs_reader as snr

## A — Load simnibs sim directories

In [2]:
# Segmentation folder (m2m_<subID>/)
m2m = snr.segmentation('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/m2m_0001')

# Simulation folder (contains mni_volumes/, subject_volumes/, ...)
sim = snr.simulation('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c')

In [ ]:
print(sim)                   # SimulationResult('...')
print(sim.tree())            # pretty directory tree

In [4]:
print(sim.sim_id)           # e.g. '0001_TDCS_1'
print(sim.available_fields)  # {'mni': [...], 'native': [...]}

0002_TDCS_1
{'mni': [PosixPath('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c/mni_volumes/0002_TDCS_1_scalar_MNI_E.nii.gz'), PosixPath('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c/mni_volumes/0002_TDCS_1_scalar_MNI_J.nii.gz'), PosixPath('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c/mni_volumes/0002_TDCS_1_scalar_MNI_magnE.nii.gz'), PosixPath('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c/mni_volumes/0002_TDCS_1_scalar_MNI_magnE_extra_fef_cleaned.nii.gz'), PosixPath('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c/mni_volumes/0002_TDCS_1_scalar_MNI_magnE_extra_fef_masked.nii.gz'), PosixPath('/Users/hippolyte.dreyfus/Documents/0-tmp-simnibs-output/simulation_simulation_fef_hemianotacs_5626899c/mni_volumes/0

## B — Access an e-field NIfTI 

In [8]:
efield = sim.magnE           # MNI space  (*_MNI_magnE.nii.gz)
# efield = sim.magnE_native    # subject space (*_magnE.nii.gz)
# efield = sim.magnJ           # current density magnitude (MNI)

print(efield)                    # EField('..._MNI_magnE.nii.gz')
print(efield.shape)              # (182, 218, 182)
print(efield.affine)             # 4x4 affine matrix
data = efield.img              # np.ndarray float32 — loaded here

EField('0002_TDCS_1_scalar_MNI_magnE.nii.gz')
(182, 218, 182, 1)
[[  -1.    0.    0.   90.]
 [   0.    1.    0. -126.]
 [   0.    0.    1.  -72.]
 [   0.    0.    0.    1.]]


## C — Extract an ROI

In [11]:
# Option 1 : existing binary NIfTI mask
roi = efield.get_roi(mask='/Volumes/levy/raw/valerocabre/hemianotACS/Data/derivatives/mri/0-lesion-synthstroke-masks-SS/sub-0001/T1_brain_lesion.nii.gz')

print(roi)           # ROI(n_voxels=523, mean=0.2341)
print(roi.stats())   # {'mean': ..., 'median': ..., 'std': ..., ...}

ROI(n_voxels=6854, status=raw)
{'mean': 0.009626840327634141, 'median': 0.003922968870028853, 'std': 0.010614393006305135, 'min': 0.0, 'max': 0.041234903037548065, 'p5': 0.0, 'p95': 0.025510870665311807, 'n_voxels': 6854, 'volume_mm3': 6854.0}


In [12]:
# Option 2 : sphere defined by MNI coordinates
roi = efield.get_roi(coords=[28, -8, 54], radius=5.0)


print(roi)           # ROI(n_voxels=523, mean=0.2341)
print(roi.stats())   # {'mean': ..., 'median': ..., 'std': ..., ...}

ROI(n_voxels=515, status=raw)
{'mean': 0.06429590157680191, 'median': 0.07363174110651016, 'std': 0.03636494892599051, 'min': 0.0, 'max': 0.11731868237257004, 'p5': 0.0, 'p95': 0.10896883383393287, 'n_voxels': 515, 'volume_mm3': 515.0}


In [13]:
# Option 3 : atlas parcel (Phase 2 — not yet implemented)
roi = efield.get_roi(atlas='harvard-oxford', region='Frontal Pole')

print(roi)           # ROI(n_voxels=523, mean=0.2341)
print(roi.stats())   # {'mean': ..., 'median': ..., 'std': ..., ...}

[fetch_atlas_harvard_oxford] Dataset found in /Users/hippolyte.dreyfus/nilearn_data/fsl

ROI(n_voxels=120948, status=raw)
{'mean': 0.017729241376773848, 'median': 0.009087082464247942, 'std': 0.026646967509852015, 'min': 0.0, 'max': 0.19026166200637817, 'p5': 0.0, 'p95': 0.07419009022414683, 'n_voxels': 120948, 'volume_mm3': 120948.0}


## D — Post-process

In [14]:
cleaned = roi.postprocess(
    smooth_fwhm=2.0,         # Gaussian smoothing in mm (None to skip)
    outlier_method='iqr',    # 'iqr' or 'z'
    portion=0.5,            # e.g. 0.95 to trim to central 95% 
)

print(cleaned)           # CleanedResult(n_kept=501, mean=0.2289)
print(cleaned.stats())   # same keys as roi.stats()

TypeError: input should be a NiftiLike object or an iterable of NiftiLike object. Got: EField

## E — Save

In [ ]:
# Save stats to TSV
roi.save(
    'results/sub01_fef_roi',
    metrics=['mean', 'median', 'std'],
    format='tsv',
)

# Save cleaned stats
cleaned.save(
    'results/sub01_fef_cleaned',
    metrics=['mean', 'median', 'std', 'n_voxels'],
    format='tsv',
)

# Save the cleaned NIfTI volume
# roi.save_nifti('results/sub01_fef.nii.gz') #TOFIX
# Save the cleaned NIfTI volume
cleaned.save_nifti('results/sub01_fef_cleaned.nii.gz')

PosixPath('results/sub01_fef_cleaned.nii.gz')

## attach a seg-results to a simu-results folder

In [ ]:
# m2m = SegmentationResult("m2m_sub01/")
# sim = SimulationResult("sim_sub01_TDCS/")
# sim.set_segmentation(m2m)  # une seule fois

sim.set_segmentation(m2m)  # une seule fois

roi_gm = (
    sim.magnE_native
       .get_roi(coords=[40, 0, 0], radius=10)
       .filter_tissue("Gray-Matter")  # label_img résolu automatiquement
)

In [ ]:
# Avec segmentation attachée (auto-résolution du brain mask)
sim.set_segmentation(m2m)
roi = sim.magnE_native.get_roi(coords=[x, y, z], radius=10)
extra = roi.complement()           # tout le cerveau SAUF la sphère
ratio = roi.stats()["mean"] / extra.stats()["mean"]

# Ou avec brain mask explicite
extra = roi.complement(brain_mask="path/to/brain_mask.nii.gz")

## Go further with:

* tutorial with freesurfer-nilearn API to extract more specifically
* tutorial with simnibs-analyze to automatized basic stats